# Bank analytics: interactive PySpark demo

Run this notebook on the `bank-analytics-jupyter` Dataproc cluster. It downloads the demo pipeline from GCS and executes it on this cluster, then reads the BigQuery results. The demo extracts are synthetic.

Run the processing cell once per day, in order: `day0`, then `day1`, then `day2`. Set `DAY` in the cell before running it. Avoid rerunning a completed day unless you are intentionally testing replay behavior.

In [ ]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName('bank-analytics-notebook').getOrCreate()
print('Spark version:', spark.version)


In [ ]:
# Download the editable pipeline file to this notebook's working directory.
import subprocess, sys, runpy
subprocess.run(['gsutil', 'cp', 'gs://id-bank-analytics/code/spark/pipeline.py', 'pipeline.py'], check=True)
print('Downloaded pipeline.py; edit it from JupyterLab before running the next cell if needed.')


In [ ]:
# Run exactly one day per execution. Change DAY to day0, day1, or day2.
DAY = 'day0'
assert DAY in {'day0', 'day1', 'day2'}
sys.argv = [
    'pipeline.py',
    '--input-uri', f'gs://id-bank-analytics/inbound/{DAY}/*.txt',
    '--config-uri', 'gs://id-bank-analytics/config/entities.json',
    '--project', 'mygcpproject-515308',
    '--dataset', 'bank_analytics',
    '--batch-id', DAY,
    '--full-refresh', 'true' if DAY == 'day0' else 'false',
    '--temporary-gcs-bucket', 'id-bank-analytics',
]
runpy.run_path('pipeline.py', run_name='__main__')
print(f'Completed {DAY}')


In [ ]:
# Recreate Spark after pipeline.py calls spark.stop(), then inspect BigQuery outputs.
spark = SparkSession.builder.appName('bank-analytics-inspection').getOrCreate()
for table in ['silver_customers', 'silver_accounts', 'silver_transactions', 'fact_transaction']:
    df = spark.read.format('bigquery').option('table', f'mygcpproject-515308.bank_analytics.{table}').load()
    print(table, df.count())
    df.show(5, truncate=False)
